In [0]:
from pyspark.sql.functions import col, current_timestamp

SCHEMA = "smart_telemetry_electricity_usage_pricing.default"
EH_NAMESPACE = "entsoeelectricityusagepricing"
SECRET_SCOPE = "kv-electricity-pricing"
CHECKPOINT_ROOT = "/Volumes/smart_telemetry_electricity_usage_pricing/default/bronze_checkpoints"

# A volume of its own: the billing notebook deletes its 'checkpoints' volume on every run.
spark.sql(f"CREATE VOLUME IF NOT EXISTS {SCHEMA}.bronze_checkpoints")

def kafka_options(hub, secret_key):
    conn = dbutils.secrets.get(SECRET_SCOPE, secret_key)
    jaas = ('kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required '
            f'username="$ConnectionString" password="{conn}";')
    return {
        "kafka.bootstrap.servers": f"{EH_NAMESPACE}.servicebus.windows.net:9093",
        "kafka.sasl.mechanism": "PLAIN",
        "kafka.security.protocol": "SASL_SSL",
        "kafka.sasl.jaas.config": jaas,
        "subscribe": hub,
        "startingOffsets": "earliest",   # used only on the very first run; afterwards the checkpoint decides
    }

def ingest_to_bronze(hub, secret_key, table):
    raw = spark.readStream.format("kafka").options(**kafka_options(hub, secret_key)).load()
    bronze = raw.select(
        col("value").cast("string").alias("body"),     # the raw JSON, untouched
        col("topic").alias("event_hub"),
        col("partition"),
        col("offset"),                                 # partition + offset identify an event uniquely
        col("timestamp").alias("enqueued_time"),       # when Event Hubs received it
        current_timestamp().alias("ingested_at"),      # when we loaded it
    )
    return (bronze.writeStream
            .format("delta")
            .outputMode("append")                       # never overwrite, never drop
            .trigger(availableNow=True)                 # read what's there, then stop
            .option("checkpointLocation", f"{CHECKPOINT_ROOT}/{table}")
            .toTable(f"{SCHEMA}.{table}"))

q1 = ingest_to_bronze("smart-meter-telemetry", "eh-telemetry-listen-connection", "bronze_meter_telemetry")
q2 = ingest_to_bronze("day-ahead-pricing", "eh-pricing-listen-connection", "bronze_day_ahead_pricing")
q1.awaitTermination()
q2.awaitTermination()

In [0]:
%sql
SELECT 'telemetry' AS source, COUNT(*) AS events, MIN(enqueued_time) AS first_event,
       MAX(enqueued_time) AS last_event, MAX(ingested_at) AS last_load
FROM smart_telemetry_electricity_usage_pricing.default.bronze_meter_telemetry
UNION ALL
SELECT 'pricing', COUNT(*), MIN(enqueued_time), MAX(enqueued_time), MAX(ingested_at)
FROM smart_telemetry_electricity_usage_pricing.default.bronze_day_ahead_pricing

source,events,first_event,last_event,last_load
telemetry,450000,2026-10-30T03:55:23.035Z,2026-10-31T03:05:47.602Z,2026-10-31T03:37:14.564Z
pricing,381,2026-10-31T02:35:22.316Z,2026-10-31T02:35:22.316Z,2026-10-31T03:37:14.776Z


In [0]:
%sql
SELECT body, partition, offset, enqueued_time
FROM smart_telemetry_electricity_usage_pricing.default.bronze_day_ahead_pricing
ORDER BY enqueued_time
LIMIT 5

body,partition,offset,enqueued_time
"{""timestamp"": ""2026-10-30 23:45:00+00:00"", ""value"": 99.9, ""currency"": ""EUR"", ""price_unit"": ""MWH"", ""eic_bidding_zone"": ""SE1""}",0,3,2026-10-31T02:35:22.316Z
"{""timestamp"": ""2026-10-30 23:15:00+00:00"", ""value"": 101.43, ""currency"": ""EUR"", ""price_unit"": ""MWH"", ""eic_bidding_zone"": ""SE1""}",0,1,2026-10-31T02:35:22.316Z
"{""timestamp"": ""2026-10-30 23:00:00+00:00"", ""value"": 104.88, ""currency"": ""EUR"", ""price_unit"": ""MWH"", ""eic_bidding_zone"": ""SE1""}",0,0,2026-10-31T02:35:22.316Z
"{""timestamp"": ""2026-10-30 23:30:00+00:00"", ""value"": 100.04, ""currency"": ""EUR"", ""price_unit"": ""MWH"", ""eic_bidding_zone"": ""SE1""}",0,2,2026-10-31T02:35:22.316Z
"{""timestamp"": ""2026-10-31 00:00:00+00:00"", ""value"": 102.69, ""currency"": ""EUR"", ""price_unit"": ""MWH"", ""eic_bidding_zone"": ""SE1""}",0,4,2026-10-31T02:35:22.316Z


In [0]:
%sql
SELECT partition, offset, COUNT(*) AS copies
FROM smart_telemetry_electricity_usage_pricing.default.bronze_meter_telemetry
GROUP BY partition, offset HAVING COUNT(*) > 1

partition,offset,copies
